## Cleaning the Final Dataset

### Standardized Data Format & Extract JSON

In [16]:
import os
import pandas as pd
import json
import re

# Paths
BASE_FOLDER = r"C:\Users\Acer\Downloads\COMP4501-FYP"
DATASET_FOLDER = os.path.join(BASE_FOLDER, "Dataset-Final")
CLEAN_FOLDER = os.path.join(BASE_FOLDER, "cleaned_dataset_final")

os.makedirs(CLEAN_FOLDER, exist_ok=True)

# ---------- Helper Functions ---------- #

def safe_json_loads(x):
    if pd.isna(x) or x == "":
        return []
    try:
        return json.loads(x)
    except:
        return []

def extract_number(text):
    if pd.isna(text):
        return None
    text = str(text).replace(",", "")
    match = re.findall(r"\d+\.?\d*", text)
    return float(match[0]) if match else None

def parse_count(text):
    if pd.isna(text):
        return None
    text = str(text).lower().replace(",", "")
    
    if 'k' in text:
        return int(float(text.replace('k','')) * 1000)
    elif 'm' in text:
        return int(float(text.replace('m','')) * 1_000_000)
    else:
        try:
            return int(float(text))
        except:
            return None

def parse_percentage(text):
    if pd.isna(text):
        return None
    val = re.sub(r"[^\d]", "", str(text))
    return int(val) if val else None

def parse_chat_time(text):
    if pd.isna(text):
        return None

    text = str(text).lower()

    if "menit" in text:
        return 0.5
    elif "jam" in text:
        return 1
    else:
        return extract_number(text)

# ---------- Feature Extraction ---------- #

def extract_product_specs(df, na_threshold=0.9):
    specs_expanded = []

    for _, row in df.iterrows():
        specs = safe_json_loads(row.get("Product Specifications", ""))

        spec_dict = {}
        for item in specs:
            name = item.get("name")
            value = item.get("value")

            if name and value:
                spec_dict[name.strip()] = value.strip()

        specs_expanded.append(spec_dict)

    specs_df = pd.DataFrame(specs_expanded)

    if not specs_df.empty:
        na_ratio = specs_df.isna().mean()
        cols_to_keep = na_ratio[na_ratio < na_threshold].index
        specs_df = specs_df[cols_to_keep]

    return pd.concat([df, specs_df], axis=1)


def extract_variations_count(df):
    counts = []

    for x in df.get("variations", []):
        data = safe_json_loads(x)

        count = 0
        for item in data:
            count += len(item.get("variations", []))

        counts.append(count)

    df["variations_count"] = counts
    return df


def extract_voucher_status(df):
    df["voucher_status"] = df["vouchers"].apply(
        lambda x: 0 if pd.isna(x) or x == "" or x == "[]" else 1
    )
    return df


def extract_media_counts(df):
    df["image_count"] = df["image"].apply(
        lambda x: len(safe_json_loads(x)) if not pd.isna(x) else 0
    )

    df["video_count"] = df["video"].apply(
        lambda x: len(safe_json_loads(x)) if not pd.isna(x) else 0
    )

    return df


# ---------- FIXED NUMERIC CLEANING ---------- #

def clean_numeric_columns(df):
    def safe_apply(col, func):
        if col in df.columns:
            return df[col].apply(func)
        else:
            return pd.Series([None] * len(df))

    # Float
    df["rating"] = pd.to_numeric(df.get("rating"), errors="coerce")
    df["seller_rating"] = pd.to_numeric(df.get("seller_rating"), errors="coerce")

    # Integer-like
    df["review"] = safe_apply("review", parse_count)
    df["stock"] = safe_apply("stock", parse_count)
    df["favorite"] = safe_apply("favorite", parse_count)
    df["seller_products"] = safe_apply("seller_products", parse_count)
    df["seller_followers"] = safe_apply("seller_followers", parse_count)
    df["sold"] = safe_apply("sold", parse_count)

    # Prices
    df["initial_price"] = safe_apply("initial_price", extract_number)
    df["final_price"] = safe_apply("final_price", extract_number)

    # Percentage
    df["seller_chats_responded_percentage"] = safe_apply(
        "seller_chats_responded_percentage", parse_percentage
    )

    # Chat time
    df["seller_chat_time_reply"] = safe_apply(
        "seller_chat_time_reply", parse_chat_time
    )

    return df


def enforce_dtypes(df):
    """Ensure uniform data types"""
    int_cols = [
        "review", "stock", "favorite",
        "seller_products", "seller_followers", "sold",
        "seller_chats_responded_percentage"
    ]

    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("Int64")

    float_cols = [
        "rating", "seller_rating",
        "initial_price", "final_price",
        "seller_chat_time_reply"
    ]

    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def calculate_gmv(df):
    df["gmv_cal"] = df["final_price"] * df["sold"]
    return df


# ---------- Main Processing ---------- #

for file in os.listdir(DATASET_FOLDER):
    if file.endswith(".csv"):
        file_path = os.path.join(DATASET_FOLDER, file)
        print(f"Processing: {file}")

        df = pd.read_csv(file_path)

        # 1. Product specs
        if "Product Specifications" in df.columns:
            df = extract_product_specs(df)

        # 2. Variations
        if "variations" in df.columns:
            df = extract_variations_count(df)

        # 3. Voucher
        if "vouchers" in df.columns:
            df = extract_voucher_status(df)

        # 4. Media
        if "image" in df.columns and "video" in df.columns:
            df = extract_media_counts(df)

        # 5. Numeric cleaning (SAFE)
        df = clean_numeric_columns(df)

        # 6. Enforce types (IMPORTANT)
        df = enforce_dtypes(df)

        # 7. GMV
        df = calculate_gmv(df)

        # Save
        save_path = os.path.join(CLEAN_FOLDER, file.replace(".csv", "_cleaned.csv"))
        df.to_csv(save_path, index=False)

        print(f"Saved to: {save_path}")

print("✅ All files processed successfully.")

Processing: Dompet_updated.csv
Saved to: C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\Dompet_updated_cleaned.csv
Processing: KaosKaki_updated.csv
Saved to: C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\KaosKaki_updated_cleaned.csv
Processing: Panci_updated.csv
Saved to: C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\Panci_updated_cleaned.csv
Processing: Sandal_updated.csv
Saved to: C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\Sandal_updated_cleaned.csv
Processing: Sunscreen_updated.csv
Saved to: C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\Sunscreen_updated_cleaned.csv
✅ All files processed successfully.


In [15]:
df_dompet = pd.read_csv(r"C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\Dompet_updated_cleaned.csv")


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)   # allow pandas to use full width

df_dompet

,url,id,title,rating,reviews,initial_price,final_price,currency,Protection,Delivery,Color,Size,stock,favorite,image,video,seller_name,shop_url,breadcrumb,Product Specifications,Product Description,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,variations,domain,brand,category_id,flash_sale,flash_sale_time,product_variation,gmv_cal,category_url,vouchers,is_available,seller_id,product_ratings,Motif,Jenis Garansi,Bahan,Masa Garansi,Negara Asal,Penutup Dompet,Jumlah Slot Kartu,Slot Koin,Tekstur Kulit,Tampilan Kulit,Jenis Kulit,Stok,Dikirim Dari,variations_count,voucher_status,image_count,video_count,review
0,https://shopee.co.id/VONA-Dompet-Pria-Lipat-Pendek-2-Premium-Kulit-PU-JERICHO-i.42592648.20460325116,20460325116,VONA Dompet Pria Lipat Pendek 2 Premium Kulit PU - JERICHO,4.79,126,145000.0,53000.0,IDR,NaN,NaN,NaN,NaN,19259,71,"[""https://down-id.img.susercontent.com/file/sg-11134201-22110-d7iyd7g3wijv6b_tn"",""https://down-id.img.susercontent.com/file/9ed6a8e5a5d4f948cb2634c4293546e2_tn"",""https://down-id.img.susercontent.com/file/sg-11134201-22110-d7iyd7g3wijv6b_tn"",""https://down-id.img.susercontent.com/file/sg-11134201-22110-vapq12l3wijvcb_tn"",""https://down-id.img.susercontent.com/file/sg-11134201-22110-o1414mq3wijv2d_tn"",""https://down-id.img.susercontent.com/file/sg-11134201-22110-1da10qu3wijv82_tn"",""https://deo.shopeemobile.com/shopee/shopee-pcmall-live-sg/5222d4ab0d91a1eca7959754168d7593.png"",""https://deo.shopeemobile.com/shopee/shopee-pcmall-live-sg/6c502a2641457578b0d5f5153b53dd5d.png"",""https://deo.shopeemobile.com/shopee/shopee-pcmall-live-sg/511aca04cc3ba9234ab0e4fcf20768a2.png"",""https://deo.shopeemobile.com/shopee/shopee-pcmall-live-sg/16ead7e0a68c3cff9f32910e4be08122.png""]","[""https://cvf.shopee.co.id/file/5da2369aaf1b59bd89f66df2bc89a544""]",VONA Official Shop,https://shopee.co.id/vona.id,"[""Shopee"",""Men Bags"",""Wallets"",""Bifold & Trifold Wallets""]","[{""name"":""Merek""},{""name"":""Motif"",""value"":""Polos""},{""name"":""Jenis Garansi"",""value"":""Tidak Garansi""},{""name"":""Bahan"",""value"":""Kulit Sintetis""},{""name"":""Masa Garansi"",""value"":""Tidak Garansi""},{""name"":""Negara Asal"",""value"":""Indonesia""},{""name"":""Penutup Dompet"",""value"":""Lipat""},{""name"":""Jumlah Slot Kartu"",""value"":""6""},{""name"":""Model Dompet"",""value"":""Slot Uang Tunai""},{""name"":""Slot Koin"",""value"":""Tidak""},{""name"":""Tekstur Kulit"",""value"":""Halus""},{""name"":""Tampilan Kulit"",""value"":""Mengkilap""},{""name"":""Jenis Kulit"",""value"":""Kulit Sintesis""},{""name"":""Stok"",""value"":""19259""},{""name"":""Dikirim Dari"",""value"":""KOTA JAKARTA SELATAN""}]","⭐ VONA Jericho : Dompet Pria Kulit PU Lipat Dua.\n⭐ Bahan Kulit PU Premium.\n⭐ Dimensi : 11 x 2 x 9 cm.\n⭐ 6 slot kartu di dalam & 1 slot Foto.\n⭐ 2 slot uang kertas & 1 slot luar.\n⭐ Desain trendy & maskulin.\n⭐ Bahan lembut, halus dan tebal.\n⭐ Jahitan rapi, kuat dan awet.\n⭐ Foto 100% Asli dari produknya.\n⭐ Merk Resmi Terdaftar di Indonesia.\n⭐ Tersedia 3 pilhan warna : Black, Coffee, Navy dan Tan.\n\nQuality Disclaimers\n Foto produk diambil langsung dari produk sesungguhnya. Pencahayaan studio foto berbeda dengan cahaya ruangan biasa, sehingga bisa tampak sedikit perbedaan warna produk. Pada foto produk, produk diisi sehingga terlihat bervolume. Saat pengiriman produk dilipat dan dalam kondisi kosong, supaya Pelanggan tidak membayar ongkos kirim yang tinggi. Silakan lihat semua foto produk (geser kanan kiri) dan pelajari informasi produk dengan seksama.\n\n\nCopyright & Trademark Notice\nVONA adalah merek resmi terdaftar yang dilindungi sepenuhnya oleh Direktorat Jenderal Kekayaan Intellektual (DJKI) Kementrian Hukum & HAM Republik Indonesia. Seluruh foto/gambar produk dan informasi produk yang tercantum pada halaman produk ini merupakan milik VONA Indonesia yang hak-haknya dilindungi oleh DJKI. Pencurian foto/gambar dan informas

### Drop Unnecessary Columns

In [20]:
import os
import pandas as pd

# Paths
BASE_FOLDER = r"C:\Users\Acer\Downloads\COMP4501-FYP"
INPUT_FOLDER = os.path.join(BASE_FOLDER, "cleaned_dataset_final")


# ---------- Final Cleanup Function ---------- #

def final_cleanup(df, na_threshold=0.98):
    """
    Final cleaning:
    - Create description_length
    - Drop sparse columns (<2% non-NA), except whitelist
    - Drop unnecessary columns
    """

    # 1. Description Length
    if "product description" in df.columns:
        df["description_length"] = df["product description"].apply(
            lambda x: len(str(x)) if not pd.isna(x) else 0
        )

    # 2. Drop Sparse Columns (<2% non-NA)
    whitelist = ["brand", "voucher", "voucher_status"]

    non_na_ratio = df.notna().mean()

    cols_to_keep = [
        col for col in df.columns
        if (non_na_ratio[col] >= (1 - na_threshold)) or (col in whitelist)
    ]

    df = df[cols_to_keep]

    # 3. Drop Explicit Columns
    cols_to_drop = [
        "url",
        "currency",
        "image",
        "video",
        "shop_url",
        "Product Specifications",
        "Product Description",
        "variations"
    ]

    df = df.drop(
        columns=[col for col in cols_to_drop if col in df.columns],
        errors="ignore"
    )

    return df


# ---------- Loop + Overwrite ---------- #

for file in os.listdir(INPUT_FOLDER):
    if file.endswith(".csv"):
        file_path = os.path.join(INPUT_FOLDER, file)
        print(f"Processing: {file}")

        try:
            df = pd.read_csv(file_path)

            # Apply final cleanup
            df = final_cleanup(df)

            # Overwrite file
            df.to_csv(file_path, index=False)

            print(f"✅ Overwritten: {file}")

        except Exception as e:
            print(f"❌ Error processing {file}: {e}")

print("🎯 All files processed and overwritten successfully.")

Processing: Dompet_updated_cleaned.csv
✅ Overwritten: Dompet_updated_cleaned.csv
Processing: KaosKaki_updated_cleaned.csv
✅ Overwritten: KaosKaki_updated_cleaned.csv
Processing: Panci_updated_cleaned.csv
✅ Overwritten: Panci_updated_cleaned.csv
Processing: Sandal_updated_cleaned.csv
✅ Overwritten: Sandal_updated_cleaned.csv
Processing: Sunscreen_updated_cleaned.csv
✅ Overwritten: Sunscreen_updated_cleaned.csv
🎯 All files processed and overwritten successfully.


In [21]:
df_dompet = pd.read_csv(r"C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\Dompet_updated_cleaned.csv")


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)   # allow pandas to use full width

df_dompet

,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,breadcrumb,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,brand,gmv_cal,Motif,Jenis Garansi,Bahan,Masa Garansi,Negara Asal,Penutup Dompet,Jumlah Slot Kartu,Slot Koin,Tekstur Kulit,Tampilan Kulit,Jenis Kulit,Stok,Dikirim Dari,variations_count,voucher_status,image_count,video_count,review
0,20460325116,VONA Dompet Pria Lipat Pendek 2 Premium Kulit PU - JERICHO,4.79,126,145000.0,53000.0,19259,71,VONA Official Shop,"[""Shopee"",""Men Bags"",""Wallets"",""Bifold & Trifold Wallets""]",4.76,111,100,1.0,6 tahun lalu,113600,281,NaN,14893000.0,Polos,Tidak Garansi,Kulit Sintetis,Tidak Garansi,Indonesia,Lipat,6,Tidak,Halus,Mengkilap,Kulit Sintesis,19259.0,KOTA JAKARTA SELATAN,4,0,10,1,0
1,4116469787,AIRA WALLET JIMS HONEY Dompet Panjang Mewah Wanita Cewek Elegant Import JH Ori Murah,4.82,174,108000.0,75000.0,0,160,JIMS HONEY Deore.id,"[""Shopee"",""Women Bags"",""Wallets"",""Long Wallets""]",4.90,419,96,1.0,6 tahun lalu,107500,304,NaN,22800000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,0,8,1,0
2,13197858916,[Lms] Dompet Pria Impor Kulit PU Premium Resleting Original MenBense,4.88,169,89000.0,39800.0,107,150,LittlemaxShop,"[""Shopee"",""Men Bags"",""Wallets"",""Bifold & Trifold Wallets""]",4.80,100,98,1.0,3 tahun lalu,3800,302,NaN,12019600.0,Embos,NaN,PU,NaN,China,Resleting,6,Ya,Timbul,Matte,PU Leather,107.0,KOTA JAKARTA BARAT,4,0,8,0,0
3,11706914967,Baellerry RFID Anti Theft Card Dompet Kartu Wallet Kulit Import BLY26,4.83,1886,51000.0,42500.0,188,23,MEGA JAYA,"[""Shopee"",""Women Bags"",""Wallets"",""Card Holders""]",4.82,839,99,1.0,7 tahun lalu,98500,3500,NaN,148750000.0,Polos,Tidak Garansi,Kulit Sintetis,NaN,China,Lainnya,8,NaN,Halus,Matte,Lainnya,188.0,KOTA JAKARTA BARAT,5,0,8,1,0
4,3406406326,Wallts Aster - Dompet Wallet Kartu Pria dan Wanita,4.93,2872,259900.0,125000.0,10093,47,Wallts Wallet Goods Official Shop,"[""Shopee"",""Men Bags"",""Wallets"",""Bifold & Trifold Wallets""]",4.91,536,100,1.0,8 tahun lalu,382300,5300,NaN,662500000.0,NaN,NaN,Kanvas,NaN,Indonesia,NaN,NaN,NaN,NaN,NaN,NaN,10093.0,KOTA BANDUNG,8,0,10,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8233,21192010627,"SHEILA POUCH By Biru Tsabita, Dompet Koin Lucu Anti Air",5.00,2,30000.0,27300.0,15980,0,TOKKO TAS MURAH,"[""Shopee"",""Women Bags"",""Wallets"",""Coin Holders & Purses""]",4.88,82,68,1.0,6 tahun lalu,6100,4,NaN,109200.0,"Logo, Polos",Garansi Supplier,NaN,24 jam,NaN,NaN,NaN,NaN,NaN,Mengkilap,Kulit Sapi,15980.0,KAB. TEGAL,16,0,8,1,0
8234,14755395784,HEYLOOK Official - Dompet Pria Panjang Hampton Dompet Cordura Panjang Lipat Premium Pria Distro - BLACK,4.79,90,175000.0,70000.0,74,125,HEYLOOK Official Shop,"[""Shopee"",""Men Bags"",""Wallets"",""Long Wallets""]",4.70,520,100,1.0,8 tahun lalu,598800,196,NaN,13720000.0,Polos,Garansi Supplier,Cordura 100,1 Bulan,Indonesia,Lipat,10,NaN,NaN,NaN,NaN,74.0,KOTA TANGERANG,4,0,10,0,0
8235,20180596374,VONA Dompet Panjang Wanita Motif Croco Elegan Banyak Slot Kartu Kulit PU - UTOPIA,4.85,20,190000.0,82000.0,24849,43,VONA Official Shop,"[""Shopee"",""Women Bags"",""Wallets"",""Long Wallets""]",4.76,111,100,1.0,6 tahun lalu,113600,33,NaN,2706000.0,Croco,Tidak Garansi,Sintetis,Tidak Garansi,Indonesia,Kancing,> 10,Ya,Timbul,Mengkilap,Kulit Sintetis,24849.0,KOTA JAKARTA SELATAN,5,0,10,1,0
8236,3572559361,Dompet Kulit Canvas Distro JFR JP25 Black Original Murah,4.80,20,175000.0,69000.0,3,3,OOKAMI INDUSTRY,"[""Shopee"",""Men Bags"",""Wallets"",""Bifold & Trifold Wallets""]",4.82,207,100,1.0,4 tahun lalu,1000,25,NaN,1725000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,KOTA SURABAYA,0,0,8,0,0


In [22]:
import os
import pandas as pd
import json

# Paths
BASE_FOLDER = r"C:\Users\Acer\Downloads\COMP4501-FYP"
INPUT_FOLDER = os.path.join(BASE_FOLDER, "cleaned_dataset_final")


# ---------- Feature Engineering Function ---------- #

def process_discount_and_breadcrumb(df):
    """
    - Create discount column
    - One-hot encode breadcrumb (controlled levels)
    """

    # ---------- 1. Discount ---------- #
    if "initial_price" in df.columns and "final_price" in df.columns:
        df["discount"] = df["initial_price"] - df["final_price"]

    # ---------- 2. Breadcrumb Processing ---------- #
    def parse_breadcrumb(x):
        try:
            if pd.isna(x):
                return []
            return json.loads(x) if isinstance(x, str) else x
        except:
            return []

    if "breadcrumb" not in df.columns:
        return df

    breadcrumb_lists = df["breadcrumb"].apply(parse_breadcrumb)

    # 🔥 CONTROL DIMENSIONALITY (recommended)
    selected_levels = [1, 2]  # Only main + sub category

    unique_categories = set()

    for lst in breadcrumb_lists:
        for i in selected_levels:
            if i < len(lst):
                val = lst[i]
                if val:
                    unique_categories.add(val.strip())

    # Create one-hot columns
    for category in unique_categories:
        col_name = f"breadcrumb_{category}"

        df[col_name] = breadcrumb_lists.apply(
            lambda x: 1 if any(
                (i < len(x) and x[i] == category) for i in selected_levels
            ) else 0
        )

    return df


# ---------- Loop + Overwrite ---------- #

for file in os.listdir(INPUT_FOLDER):
    if file.endswith(".csv"):
        file_path = os.path.join(INPUT_FOLDER, file)
        print(f"Processing: {file}")

        try:
            df = pd.read_csv(file_path)

            # Apply feature engineering
            df = process_discount_and_breadcrumb(df)

            # 🔥 OVERWRITE FILE
            df.to_csv(file_path, index=False)

            print(f"✅ Overwritten: {file}")

        except Exception as e:
            print(f"❌ Error processing {file}: {e}")

print("🎯 All files updated with discount + breadcrumb encoding.")

Processing: Dompet_updated_cleaned.csv
✅ Overwritten: Dompet_updated_cleaned.csv
Processing: KaosKaki_updated_cleaned.csv
✅ Overwritten: KaosKaki_updated_cleaned.csv
Processing: Panci_updated_cleaned.csv


C:\Users\Acer\AppData\Local\Temp\ipykernel_11924\3920750094.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = breadcrumb_lists.apply(
C:\Users\Acer\AppData\Local\Temp\ipykernel_11924\3920750094.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = breadcrumb_lists.apply(
C:\Users\Acer\AppData\Local\Temp\ipykernel_11924\3920750094.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining

✅ Overwritten: Panci_updated_cleaned.csv
Processing: Sandal_updated_cleaned.csv
✅ Overwritten: Sandal_updated_cleaned.csv
Processing: Sunscreen_updated_cleaned.csv
✅ Overwritten: Sunscreen_updated_cleaned.csv
🎯 All files updated with discount + breadcrumb encoding.


In [23]:
df_dompet = pd.read_csv(r"C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\Dompet_updated_cleaned.csv")


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)   # allow pandas to use full width

df_dompet

C:\Users\Acer\AppData\Local\Temp\ipykernel_11924\2683624715.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dompet = pd.read_csv(r"C:\Users\Acer\Downloads\COMP4501-FYP\cleaned_dataset_final\Dompet_updated_cleaned.csv")


,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,breadcrumb,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,brand,gmv_cal,Motif,Jenis Garansi,Bahan,Masa Garansi,Negara Asal,Penutup Dompet,Jumlah Slot Kartu,Slot Koin,Tekstur Kulit,Tampilan Kulit,Jenis Kulit,Stok,Dikirim Dari,variations_count,voucher_status,image_count,video_count,review,discount,breadcrumb_Mobile & Tablet Accessories,breadcrumb_Automotive Keychains & Key Covers,breadcrumb_Other Fashion Accessories,breadcrumb_Tote Bags,breadcrumb_Sports & Outdoor,breadcrumb_Other Sports & Outdoor,breadcrumb_Women Shoes,breadcrumb_Tas & Koper,breadcrumb_Aksesoris Make Up,breadcrumb_Umbrella,breadcrumb_Aksesoris Bayi & Anak,breadcrumb_Waist Bags & Chest Bags,breadcrumb_Motorcycle Rider Accessories,breadcrumb_Dompet Wanita,breadcrumb_Logam Mulia,breadcrumb_Sets,breadcrumb_Tas Wanita,breadcrumb_Women Bags,breadcrumb_Aksesoris Konsol,breadcrumb_Fashion Bayi & Anak,breadcrumb_Bag Accessories,breadcrumb_Handphone & Aksesoris,breadcrumb_Boy Bags,breadcrumb_Men Shoes,breadcrumb_Dompet,breadcrumb_Alat Kecantikan,breadcrumb_Ear Care,breadcrumb_Muslim Fashion,breadcrumb_Clutches & Wristlets,breadcrumb_Home & Living,breadcrumb_Health,breadcrumb_Aksesoris Fashion Lainnya,breadcrumb_Mobile & Accessories,breadcrumb_First Aid Supplies,breadcrumb_Eyes Makeup,breadcrumb_Other Photography,breadcrumb_Needlework,breadcrumb_Souvenir & Perlengkapan Pesta,breadcrumb_Stationery & Books,breadcrumb_Beauty & Care,breadcrumb_Men Clothes,breadcrumb_Casing & Skin,breadcrumb_Tas Pria,breadcrumb_Baby & Kids Fashion,breadcrumb_Eyewear,breadcrumb_Clutch,breadcrumb_Girl Bags,breadcrumb_Crossbody & Shoulder Bags,breadcrumb_Console Accessories,breadcrumb_Fashion Accessories,breadcrumb_Home Organizers,breadcrumb_Automobile Exterior Accessories,breadcrumb_Aksesoris,breadcrumb_Automotive,breadcrumb_Other Men Bags,breadcrumb_Casing & Protectors,breadcrumb_Tools,"breadcrumb_Folders, Paper Organizers & Accessories",breadcrumb_Non-Fiction Books,breadcrumb_Shoe Care & Accessories,breadcrumb_School & Office Equipment,breadcrumb_Bath & Body Care,breadcrumb_Dompet Koin,breadcrumb_Clutches,breadcrumb_Sports & Outdoor Accessories,breadcrumb_Prayer Attire & Equipment,"breadcrumb_Kabel, Charger, & Konverter",breadcrumb_Hobby & Collection,breadcrumb_Aksesoris Fashion,breadcrumb_Dompet Lipat,"breadcrumb_Cables, Chargers & Converters",breadcrumb_Art Supplies,breadcrumb_Handphone & Aksesoris Lainnya,breadcrumb_Backpacks,breadcrumb_Tas Anak Perempuan,breadcrumb_Other Women Bags,breadcrumb_Dompet Kartu,breadcrumb_Electronics,breadcrumb_Food Supplement,breadcrumb_Wallets,breadcrumb_Souvenir & Hadiah,breadcrumb_Camera Care,breadcrumb_Dompet Kunci & Handphone,breadcrumb_Tas Selempang & Bahu Wanita,breadcrumb_Men Muslim Wear,breadcrumb_Dompet Panjang,breadcrumb_Beauty Sets & Packages,breadcrumb_Men Bags,breadcrumb_Laptop Bags,breadcrumb_Other Women Shoes,breadcrumb_Tas Anak Laki-Laki,breadcrumb_Pouch Handphone,breadcrumb_Top-handle Bags,breadcrumb_Photography,breadcrumb_Makeup Accessories
0,20460325116,VONA Dompet Pria Lipat Pendek 2 Premium Kulit PU - JERICHO,4.79,126,145000.0,53000.0,19259,71,VONA Official Shop,"[""Shopee"",""Men Bags"",""Wallets"",""Bifold & Trifold Wallets""]",4.76,111,100,1.0,6 tahun lalu,113600,281,NaN,14893000.0,Polos,Tidak Garansi,Kulit Sintetis,Tidak Garansi,Indonesia,Lipat,6,Tidak,Halus,Mengkilap,Kulit Sintesis,19259.0,KOTA JAKARTA SELATAN,4,0,10,1,0,92000.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,4116469787,AIRA WALLET JIMS HONEY Dompet Panjang Mewah Wanita Cewek Elegant Import JH Ori Murah,4.82,174,108000.0,75000.0,0,160,JIMS HONEY Deore.id,"[""Shopee"",""Women Bags"",""Wallets"",""Long Wallets""]",4.90,419,96,1.0,6 tahun lalu,107500,304,NaN,22800000.0,NaN,